# Activity: Watch NEAT grow a neural network

**Goal:** see how NEAT builds a network from nothing, then break it on purpose to find out which parts of NEAT matter.

**The task: XOR.** The network gets two inputs (0 or 1) and should output 1 only when *exactly one* input is 1. A network with no hidden neurons can't solve this, so NEAT has to invent at least one.

Work in pairs. About 20 minutes total:

1. **Setup** (2 min): run the first two code cells.
2. **Part 1** (5 min): watch one normal NEAT run.
3. **Part 2** (8 min): pick an experiment card, predict, run, report.
4. **Part 3** (5 min): class discussion.

## Setup
Run these two cells (click a cell, then press **Shift + Enter**). The second one just loads helper code, so you don't need to read it.

In [ ]:
!pip install -q neat-python

In [ ]:
# Helper code for the activity (no need to read it)
import os, random, tempfile, contextlib, io
import neat
from neat.graphs import feed_forward_layers
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

BASE = {
    # [NEAT]
    "pop_size": 150, "fitness_threshold": 3.9,
    # [DefaultGenome]
    "num_hidden": 0, "initial_connection": "full",
    "node_add_prob": 0.2, "node_delete_prob": 0.2,
    "conn_add_prob": 0.5, "conn_delete_prob": 0.5,
    "weight_mutate_rate": 0.8, "weight_mutate_power": 0.5,
    "compatibility_disjoint_coefficient": 1.0, "compatibility_weight_coefficient": 0.5,
    # [DefaultSpeciesSet]
    "compatibility_threshold": 3.0,
}

TEMPLATE = """
[NEAT]
fitness_criterion = max
fitness_threshold = {fitness_threshold}
pop_size = {pop_size}
reset_on_extinction = True
no_fitness_termination = False

[DefaultGenome]
num_inputs = 2
num_hidden = {num_hidden}
num_outputs = 1
initial_connection = {initial_connection}
feed_forward = True
activation_default = sigmoid
activation_mutate_rate = 0.0
activation_options = sigmoid
aggregation_default = sum
aggregation_mutate_rate = 0.0
aggregation_options = sum
bias_init_mean = 0.0
bias_init_stdev = 1.0
bias_max_value = 30.0
bias_min_value = -30.0
bias_mutate_power = {weight_mutate_power}
bias_mutate_rate = 0.7
bias_replace_rate = 0.1
response_init_mean = 1.0
response_init_stdev = 0.0
response_max_value = 30.0
response_min_value = -30.0
response_mutate_power = 0.0
response_mutate_rate = 0.0
response_replace_rate = 0.0
weight_init_mean = 0.0
weight_init_stdev = 1.0
weight_max_value = 30
weight_min_value = -30
weight_mutate_power = {weight_mutate_power}
weight_mutate_rate = {weight_mutate_rate}
weight_replace_rate = 0.1
enabled_default = True
enabled_mutate_rate = 0.01
conn_add_prob = {conn_add_prob}
conn_delete_prob = {conn_delete_prob}
node_add_prob = {node_add_prob}
node_delete_prob = {node_delete_prob}
compatibility_disjoint_coefficient = {compatibility_disjoint_coefficient}
compatibility_weight_coefficient = {compatibility_weight_coefficient}

[DefaultSpeciesSet]
compatibility_threshold = {compatibility_threshold}

[DefaultStagnation]
species_fitness_func = max
max_stagnation = 20
species_elitism = 2

[DefaultReproduction]
elitism = 2
survival_threshold = 0.2
"""

XOR_INPUTS = [(0.0, 0.0), (0.0, 1.0), (1.0, 0.0), (1.0, 1.0)]
XOR_OUTPUTS = [0.0, 1.0, 1.0, 0.0]


def eval_genomes(genomes, config):
    for _, genome in genomes:
        net = neat.nn.FeedForwardNetwork.create(genome, config)
        genome.fitness = 4.0
        for xi, xo in zip(XOR_INPUTS, XOR_OUTPUTS):
            genome.fitness -= (net.activate(xi)[0] - xo) ** 2


def make_config(changes=None):
    settings = dict(BASE)
    settings.update(changes or {})
    path = os.path.join(tempfile.gettempdir(), "neat_activity_config.ini")
    with open(path, "w") as f:
        f.write(TEMPLATE.format(**settings))
    return neat.Config(neat.DefaultGenome, neat.DefaultReproduction,
                       neat.DefaultSpeciesSet, neat.DefaultStagnation, path)



def run_neat(changes=None, generations=150, seed=1):
    """Run NEAT on XOR once. Returns a dict with the winner, stats, etc."""
    config = make_config(changes)
    pop = neat.Population(config, seed=seed)
    stats = neat.StatisticsReporter()
    pop.add_reporter(stats)
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            winner = pop.run(eval_genomes, generations)
    except RuntimeError:
        # neat-python stops if there are more species than the population can hold
        return {"config": config, "stats": stats, "winner": None,
                "generations": generations, "solved": False, "crashed": True}
    return {"config": config, "stats": stats, "winner": winner,
            "generations": len(stats.most_fit_genomes),
            "solved": winner.fitness >= config.fitness_threshold, "crashed": False}


def hidden_count(result):
    """How many hidden neurons the best network has."""
    if result["winner"] is None:
        return 0
    out = result["config"].genome_config.output_keys
    return sum(1 for k in result["winner"].nodes if k not in out)


def summarize(label, results):
    solved = sum(r["solved"] for r in results)
    gens = [r["generations"] for r in results if r["solved"]]
    hidden = [hidden_count(r) for r in results if r["solved"]]
    avg_g = f"{sum(gens)/len(gens):.0f}" if gens else "-"
    avg_h = f"{sum(hidden)/len(hidden):.1f}" if hidden else "-"
    print(f"{label:12s} solved {solved}/{len(results)} runs | avg generations to solve: {avg_g:>4s} | avg hidden neurons: {avg_h}")


def experiment(changes, seeds=range(1, 9), generations=150):
    """Run baseline and your change on the same 8 random seeds, then compare."""
    base = [run_neat(None, generations, s) for s in seeds]
    mine = [run_neat(changes, generations, s) for s in seeds]
    print(f"Your change: {changes}\n")
    summarize("baseline", base)
    summarize("your change", mine)
    crashed = sum(r["crashed"] for r in mine)
    if crashed:
        print(f"\n({crashed} run(s) stopped early: too many species for the population size)")
    best = max((r for r in mine if r["winner"] is not None), key=lambda r: r["winner"].fitness, default=None)
    if best is not None:
        show_run(best, title="Best network with your change")


def draw_network(result, ax):
    """Draw the winning network: inputs left, output right.
    Blue = positive weight, red = negative, thicker = stronger, dashed grey = disabled."""
    g, cfg = result["winner"], result["config"].genome_config
    inputs, outputs = list(cfg.input_keys), list(cfg.output_keys)
    enabled = [c.key for c in g.connections.values() if c.enabled]
    depth = {k: 0 for k in inputs}
    def d(n, seen=()):
        if n in depth:
            return depth[n]
        srcs = [a for a, b in enabled if b == n and a not in seen]
        depth[n] = 1 + max((d(a, seen + (n,)) for a in srcs), default=0)
        return depth[n]
    for n in g.nodes:
        d(n)
    last = max([depth[n] for n in g.nodes] + [1])
    for o in outputs:
        depth[o] = last
    cols = {}
    for n in inputs + [k for k in g.nodes if k not in outputs] + outputs:
        cols.setdefault(depth[n], []).append(n)
    pos = {}
    for x, members in cols.items():
        for i, n in enumerate(sorted(members)):
            pos[n] = (x, (i + 1) / (len(members) + 1))
    for c in g.connections.values():
        a, b = c.key
        if a not in pos or b not in pos:
            continue
        (x1, y1), (x2, y2) = pos[a], pos[b]
        if c.enabled:
            ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                        arrowprops=dict(arrowstyle="-|>", lw=min(0.6 + abs(c.weight) / 2, 4),
                                        color="tab:blue" if c.weight > 0 else "tab:red",
                                        shrinkA=14, shrinkB=14))
        else:
            ax.plot([x1, x2], [y1, y2], ls="--", color="lightgrey", zorder=0)
    for n, (x, y) in pos.items():
        kind = "x" + str(-n) if n in inputs else ("out" if n in outputs else "h")
        face = "white" if n in inputs else ("#ffe08a" if n in outputs else "#b8e3b5")
        ax.scatter([x], [y], s=900, c=face, edgecolors="black", zorder=3)
        ax.text(x, y, kind, ha="center", va="center", fontsize=10, zorder=4)
    ax.set_xlim(-0.5, last + 0.5); ax.set_ylim(0, 1); ax.axis("off")
    ax.set_title(f"Best network: {hidden_count(result)} hidden neuron(s)")


def show_run(result, title="NEAT on XOR"):
    """Fitness over time, number of species over time, and the best network."""
    stats = result["stats"]
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    best = [g.fitness for g in stats.most_fit_genomes]
    axes[0].plot(best, label="best"); axes[0].plot(stats.get_fitness_mean(), label="average")
    axes[0].axhline(3.9, ls=":", color="grey", label="solved")
    axes[0].set_xlabel("generation"); axes[0].set_ylabel("fitness (4 = perfect)"); axes[0].legend()
    axes[0].set_title("Fitness")
    sizes = stats.get_species_sizes()
    axes[1].stackplot(range(len(sizes)), list(zip(*sizes)) if sizes else [])
    axes[1].set_xlabel("generation"); axes[1].set_ylabel("networks"); axes[1].set_title("Species (each colour = one species)")
    if result["winner"] is not None:
        draw_network(result, axes[2])
    fig.suptitle(title); plt.tight_layout(); plt.show()
    if result["winner"] is not None:
        net = neat.nn.FeedForwardNetwork.create(result["winner"], result["config"])
        for xi, xo in zip(XOR_INPUTS, XOR_OUTPUTS):
            print(f"input {xi} -> expected {xo:.0f}, network says {net.activate(xi)[0]:.2f}")


## Part 1: One normal NEAT run (5 min)

Run the cell below. It evolves 150 networks at a time for up to 150 generations and stops as soon as one solves XOR.

Look at the three plots and answer with your partner:

- **Fitness:** how many generations until the best network solved it? Does the average catch up?
- **Species:** how many species were there? Do new species appear and old ones disappear?
- **Network:** how many hidden neurons did NEAT add? Remember every network *started* with zero.

In [ ]:
baseline = run_neat(seed=1)
show_run(baseline, title='Baseline NEAT on XOR')

## Part 2: Experiment cards (8 min)

Each pair gets **one card**. Each card changes one part of NEAT.

**Before you run anything,** write down a prediction: will NEAT solve XOR more often, less often, or the same? Faster or slower? More or fewer hidden neurons?

| Card | What it changes | Code to paste below |
| --- | --- | --- |
| A | NEAT can never add a new neuron | `{"node_add_prob": 0.0}` |
| B | Everything is one species, so no protection for new ideas | `{"compatibility_threshold": 100.0}` |
| C | Tiny population: 10 networks instead of 150 | `{"pop_size": 10}` |
| D | Huge random weight changes when mutating | `{"weight_mutate_power": 2.0}` |
| E | NEAT adds neurons very eagerly | `{"node_add_prob": 0.8}` |
| F | NEAT can never add or remove a connection | `{"conn_add_prob": 0.0, "conn_delete_prob": 0.0}` |

The cell runs the normal version and your version on the **same 8 random seeds**, so a difference isn't just luck. It takes up to a minute.

In [ ]:
MY_CARD = {"node_add_prob": 0.0}   # <- replace with your card's code

experiment(MY_CARD)

**Report back (one sentence each):**

1. What did you predict?
2. What happened? (solved X/8, generations, hidden neurons)
3. *Why* do you think it happened? Connect it to an idea from the NEAT paper.

## Part 3: Discussion questions

- Which card hurt NEAT the most? What does that tell you about XOR?
- Card E grew lots of neurons. Why might a bigger network be *worse*, even if it solves the problem?
- Card B removed speciation. What does that tell you about why the NEAT paper protects new structure?
- Card D is a lot like setting the learning rate too high in gradient descent. What's the connection?
- NEAT never computed a single gradient here. What did it use instead?

**Bonus if you finish early:** make your own card by combining two changes, or try a different seed with `run_neat(MY_CARD, seed=5)`.